# Training Classification Models

In this notebook, we explore several approaches using Machine Learning, Deep Learning, and even BERT models to try to classify competencies based on course descriptions.

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.multioutput import ClassifierChain
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import random
import sys
import time

from pipeline_utils import (
    DEFAULT_K_VALUES,
    evaluate_baselines,
    compute_multilabel_metrics,
    export_experiment_artifacts,
)

# --- 1. Seed Configuration ---
def set_seed(seed_value=42):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    random.seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- 2. MLP Wrapper to Look Like Scikit-Learn ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        BCE_loss = nn.BCEWithLogitsLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-BCE_loss)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        F_loss = alpha_t * (1 - pt) ** self.gamma * BCE_loss
        return torch.mean(F_loss) if self.reduction == 'mean' else torch.sum(F_loss)

class PyTorchMLPWrapper:
    def __init__(self, input_dim, num_labels, epochs=50, batch_size=16, lr=1e-4):
        self.input_dim = input_dim
        self.num_labels = num_labels
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_labels)
        ).to(self.device)

    def fit(self, X, y):
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32).to(self.device)
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        criterion = FocalLoss()
        optimizer = optim.AdamW(self.model.parameters(), lr=self.lr)
        self.model.train()
        for epoch in range(self.epochs):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        loader = DataLoader(TensorDataset(X_tensor), batch_size=self.batch_size, shuffle=False)
        all_probs = []
        with torch.no_grad():
            for batch in loader:
                logits = self.model(batch[0])
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
        return np.concatenate(all_probs, axis=0)


def _score_matrix_from_predict_proba(y_proba, n_labels):
    if isinstance(y_proba, list):
        return np.column_stack([p[:, 1] if p.ndim == 2 and p.shape[1] > 1 else p.ravel() for p in y_proba])
    y_proba = np.asarray(y_proba, dtype=float)
    if y_proba.ndim == 3:
        y_proba = y_proba[:, :, 1]
    if y_proba.ndim == 1:
        y_proba = y_proba.reshape(-1, 1)
    if y_proba.shape[1] != n_labels:
        raise ValueError(f"Probability shape {y_proba.shape} does not match {n_labels} labels.")
    return y_proba


def evaluate_and_export(method, embedding_type, y_test, y_score, classes, timing_rows, config):
    metrics_df, predictions_df = compute_multilabel_metrics(
        y_test,
        y_score,
        labels=classes,
        method=method,
        k_values=DEFAULT_K_VALUES,
    )
    print(metrics_df)
    paths = export_experiment_artifacts(
        results_dir="results/ml_models",
        method=f"{method}_{embedding_type}",
        metrics_df=metrics_df,
        predictions_df=predictions_df,
        timing_rows=timing_rows,
        config=config,
    )
    print("Exported artifacts:", paths)
    return metrics_df


def run_baselines(y_train, y_test, classes, embedding_type):
    metrics_df, predictions_df, timing_df = evaluate_baselines(
        y_train,
        y_test,
        labels=classes,
        k_values=DEFAULT_K_VALUES,
        seed=42,
    )
    export_experiment_artifacts(
        results_dir="results/ml_models",
        method=f"baselines_{embedding_type}",
        metrics_df=metrics_df,
        predictions_df=predictions_df,
        timing_rows=timing_df.to_dict("records"),
        config={
            "notebook": "ml_models.ipynb",
            "embedding_type": embedding_type,
            "seed": 42,
            "k_values": DEFAULT_K_VALUES,
            "labels": list(classes),
            "n_train": int(y_train.shape[0]),
            "n_test": int(y_test.shape[0]),
            "baselines": ["frequency_topk", "random_distribution"],
        },
    )
    print(metrics_df)
    return metrics_df


def run_experiment(model_name, X_train, y_train, X_test, y_test, mlb_classes, embedding_type="unknown"):
    print(f"\n{'='*40}\nRunning: {model_name} ({embedding_type})\n{'='*40}")
    if model_name == 'xgb':
        base = XGBClassifier(n_estimators=250, max_depth=5, random_state=42, eval_metric='logloss')
        model = ClassifierChain(base, order='random', random_state=42)
    elif model_name == 'rf':
        base = RandomForestClassifier(n_estimators=250, random_state=42)
        model = ClassifierChain(base, order='random', random_state=42)
    elif model_name == 'gb':
        base = GradientBoostingClassifier(n_estimators=250, random_state=42)
        model = ClassifierChain(base, order='random', random_state=42)
    elif model_name == 'mlp':
        model = PyTorchMLPWrapper(input_dim=X_train.shape[1], num_labels=y_train.shape[1])
    elif model_name == 'br_lr':
        model = OneVsRestClassifier(LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    else:
        raise ValueError("Unknown model. Use: 'xgb', 'rf', 'gb', 'mlp', or 'br_lr'")

    train_start = time.perf_counter()
    model.fit(X_train, y_train)
    train_seconds = time.perf_counter() - train_start

    infer_start = time.perf_counter()
    y_proba = model.predict_proba(X_test)
    inference_seconds = time.perf_counter() - infer_start
    y_score = _score_matrix_from_predict_proba(y_proba, y_train.shape[1])

    timing_rows = [{
        "method": model_name,
        "embedding_type": embedding_type,
        "train_seconds": train_seconds,
        "inference_seconds": inference_seconds,
        "inference_seconds_per_sample": inference_seconds / max(1, len(X_test)),
    }]
    return evaluate_and_export(
        method=model_name,
        embedding_type=embedding_type,
        y_test=y_test,
        y_score=y_score,
        classes=mlb_classes,
        timing_rows=timing_rows,
        config={
            "notebook": "ml_models.ipynb",
            "method": model_name,
            "embedding_type": embedding_type,
            "seed": 42,
            "k_values": DEFAULT_K_VALUES,
            "labels": list(mlb_classes),
            "n_train": int(y_train.shape[0]),
            "n_test": int(y_test.shape[0]),
        },
    )


In [2]:
import random
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

import torch
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
from sklearn.feature_extraction.text import TfidfVectorizer

from pipeline_utils import load_split_csv, prepare_multilabel_targets

try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpus/stopwords')
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt_tab', quiet=True)

STOP_WORDS_PT = set(stopwords.words('portuguese'))
DOMAIN_STOP_WORDS = {
    'curso', 'aprendizagem', 'educação', 'gestão', 'avaliação', 'pessoas',
    'científica', 'inclusão', 'trabalho', 'ensino', 'servidores', 'uso',
    'objetivo', 'conhecimento', 'público', 'formação', 'conceitos'
}
ALL_STOP_WORDS = list(STOP_WORDS_PT.union(DOMAIN_STOP_WORDS))


def _get_tfidf_features(train_texts, test_texts):
    print("-> Generating TF-IDF features...")
    tfidf = TfidfVectorizer(
        max_features=2000, ngram_range=(1, 2), min_df=5, max_df=0.7,
        stop_words=ALL_STOP_WORDS, sublinear_tf=True
    )
    X_train = tfidf.fit_transform(train_texts).toarray()
    X_test = tfidf.transform(test_texts).toarray()
    return X_train, X_test


def _clean_text(text):
    text = text.lower().translate(str.maketrans('', '', string.punctuation))
    return [w for w in word_tokenize(text) if w not in STOP_WORDS_PT and w.isalpha()]


def _get_word2vec_features(train_texts, test_texts, vector_size=300):
    print(f"-> Generating Word2Vec features (dim={vector_size})...")
    train_tokens = [_clean_text(t) for t in train_texts]
    test_tokens = [_clean_text(t) for t in test_texts]
    model = Word2Vec(sentences=train_tokens, vector_size=vector_size, window=5, min_count=2, workers=4, seed=42)

    def embed(tokens):
        vectors = []
        for token_list in tokens:
            valid = [model.wv[w] for w in token_list if w in model.wv]
            vectors.append(np.mean(valid, axis=0) if valid else np.zeros(vector_size))
        return np.vstack(vectors)

    return embed(train_tokens), embed(test_tokens)


def _get_bert_embeddings(texts):
    print("-> Generating BERT features (this may take some time)...")
    model_name = 'neuralmind/bert-base-portuguese-cased'
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)
    batch_size = 32
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors='pt', truncation=True, padding=True, max_length=128)
        with torch.no_grad():
            outputs = model(**inputs)
        all_embeddings.append(outputs.last_hidden_state[:, 0, :].numpy())
    return np.vstack(all_embeddings)


def get_data_pipeline(embedding_type='tfidf', train_path='train.csv', test_path='test.csv'):
    train_df = load_split_csv(train_path)
    test_df = load_split_csv(test_path)
    train_texts = train_df['combinedText'].astype(str).tolist()
    test_texts = test_df['combinedText'].astype(str).tolist()

    if embedding_type == 'tfidf':
        X_train, X_test = _get_tfidf_features(train_texts, test_texts)
    elif embedding_type == 'word2vec':
        X_train, X_test = _get_word2vec_features(train_texts, test_texts, vector_size=300)
    elif embedding_type == 'bert':
        X_train = _get_bert_embeddings(train_texts)
        X_test = _get_bert_embeddings(test_texts)
    else:
        raise ValueError("Embedding must be one of: 'tfidf', 'word2vec', or 'bert'")

    y_train, y_test, classes, mlb = prepare_multilabel_targets(train_df, test_df)
    print(f"Pipeline completed. X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")
    return X_train, X_test, y_train, y_test, classes


c:\Users\Administrador\.conda\envs\plaforedu-06-05\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Baseline - TF-IDF

In [3]:
# Obtain data prepared for TFIDF
embedding_type = 'tfidf'
X_train, X_test, y_train, y_test, classes = get_data_pipeline(embedding_type)
run_baselines(y_train, y_test, classes, embedding_type)

-> Generating TF-IDF features...
Pipeline completed. X_train shape: (270, 634), X_test shape: (68, 634)
                         method   k  micro_f1  macro_f1  hamming_loss  \
0       baseline_frequency_topk   1  0.181818  0.009224      0.054939   
1       baseline_frequency_topk   3  0.238095  0.019934      0.079911   
2       baseline_frequency_topk   5  0.229572  0.026978      0.109878   
3       baseline_frequency_topk   7  0.215385  0.032597      0.141509   
4       baseline_frequency_topk  10  0.187354  0.037825      0.192564   
5  baseline_random_distribution   1  0.157025  0.019749      0.056604   
6  baseline_random_distribution   3  0.164021  0.022607      0.087680   
7  baseline_random_distribution   5  0.175097  0.042265      0.117647   
8  baseline_random_distribution   7  0.175385  0.057509      0.148724   
9  baseline_random_distribution  10  0.156909  0.058278      0.199778   

   subset_accuracy  precision_at_k  recall_at_k  partial_hit_at_k      lrap  \
0            

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,baseline_frequency_topk,1,0.181818,0.009224,0.054939,0.0,0.323529,0.129797,0.323529,0.344820,23.529412,68,53
1,baseline_frequency_topk,3,0.238095,0.019934,0.079911,0.0,0.220588,0.282878,0.397059,0.344820,23.529412,68,53
2,baseline_frequency_topk,5,0.229572,0.026978,0.109878,0.0,0.173529,0.392058,0.573529,0.344820,23.529412,68,53
3,baseline_frequency_topk,7,0.215385,0.032597,0.141509,0.0,0.147059,0.443284,0.632353,0.344820,23.529412,68,53
4,baseline_frequency_topk,10,0.187354,0.037825,0.192564,0.0,0.117647,0.499796,0.705882,0.344820,23.529412,68,53
5,baseline_random_distribution,1,0.157025,0.019749,0.056604,0.0,0.279412,0.114601,0.279412,0.262877,28.161765,68,53
6,baseline_random_distribution,3,0.164021,0.022607,0.087680,0.0,0.151961,0.196954,0.367647,0.262877,28.161765,68,53
7,baseline_random_distribution,5,0.175097,0.042265,0.117647,0.0,0.132353,0.259320,0.470588,0.262877,28.161765,68,53
8,baseline_random_distribution,7,0.175385,0.057509,0.148724,0.0,0.119748,0.333836,0.573529,0.262877,28.161765,68,53
9,baseline_random_distribution,10,0.156909,0.058278,0.199778,0.0,0.098529,0.395117,0.602941,0.262877,28.161765,68,53


## Machine Learning - TF-IDF Vectorizer

In [4]:
run_experiment('br_lr', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: br_lr (tfidf)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0  br_lr   1  0.264463  0.105973      0.049390         0.088235   
1  br_lr   3  0.354497  0.186095      0.067703         0.014706   
2  br_lr   5  0.334630  0.199731      0.094895         0.000000   
3  br_lr   7  0.298462  0.184346      0.126526         0.000000   
4  br_lr  10  0.252927  0.160936      0.177026         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.470588     0.235896          0.470588  0.487043       17.529412   
1        0.328431     0.446575          0.661765  0.487043       17.529412   
2        0.252941     0.545706          0.705882  0.487043       17.529412   
3        0.203782     0.620496          0.764706  0.487043       17.529412   
4        0.158824     0.675264          0.823529  0.487043       17.529412   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3   

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,br_lr,1,0.264463,0.105973,0.049390,0.088235,0.470588,0.235896,0.470588,0.487043,17.529412,68,53
1,br_lr,3,0.354497,0.186095,0.067703,0.014706,0.328431,0.446575,0.661765,0.487043,17.529412,68,53
2,br_lr,5,0.334630,0.199731,0.094895,0.000000,0.252941,0.545706,0.705882,0.487043,17.529412,68,53
3,br_lr,7,0.298462,0.184346,0.126526,0.000000,0.203782,0.620496,0.764706,0.487043,17.529412,68,53
4,br_lr,10,0.252927,0.160936,0.177026,0.000000,0.158824,0.675264,0.823529,0.487043,17.529412,68,53


In [5]:
run_experiment('gb', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: gb (tfidf)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0     gb   1  0.198347  0.080845      0.053829         0.058824   
1     gb   3  0.275132  0.115926      0.076027         0.014706   
2     gb   5  0.249027  0.134061      0.107103         0.000000   
3     gb   7  0.227692  0.130107      0.139290         0.000000   
4     gb  10  0.199063  0.130115      0.189789         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.352941     0.181239          0.352941  0.384547       22.573529   
1        0.254902     0.339957          0.558824  0.384547       22.573529   
2        0.188235     0.405399          0.647059  0.384547       22.573529   
3        0.155462     0.480889          0.705882  0.384547       22.573529   
4        0.125000     0.545706          0.750000  0.384547       22.573529   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3      

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,gb,1,0.198347,0.080845,0.053829,0.058824,0.352941,0.181239,0.352941,0.384547,22.573529,68,53
1,gb,3,0.275132,0.115926,0.076027,0.014706,0.254902,0.339957,0.558824,0.384547,22.573529,68,53
2,gb,5,0.249027,0.134061,0.107103,0.000000,0.188235,0.405399,0.647059,0.384547,22.573529,68,53
3,gb,7,0.227692,0.130107,0.139290,0.000000,0.155462,0.480889,0.705882,0.384547,22.573529,68,53
4,gb,10,0.199063,0.130115,0.189789,0.000000,0.125000,0.545706,0.750000,0.384547,22.573529,68,53


In [6]:
run_experiment('rf', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: rf (tfidf)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0     rf   1  0.223140  0.045016      0.052164         0.058824   
1     rf   3  0.328042  0.127229      0.070477         0.000000   
2     rf   5  0.326848  0.151383      0.096004         0.000000   
3     rf   7  0.292308  0.143221      0.127636         0.000000   
4     rf  10  0.250585  0.143957      0.177580         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.397059     0.185055          0.397059  0.441464       19.808824   
1        0.303922     0.416673          0.602941  0.441464       19.808824   
2        0.247059     0.543529          0.720588  0.441464       19.808824   
3        0.199580     0.600041          0.779412  0.441464       19.808824   
4        0.157353     0.681414          0.838235  0.441464       19.808824   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3      

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,rf,1,0.223140,0.045016,0.052164,0.058824,0.397059,0.185055,0.397059,0.441464,19.808824,68,53
1,rf,3,0.328042,0.127229,0.070477,0.000000,0.303922,0.416673,0.602941,0.441464,19.808824,68,53
2,rf,5,0.326848,0.151383,0.096004,0.000000,0.247059,0.543529,0.720588,0.441464,19.808824,68,53
3,rf,7,0.292308,0.143221,0.127636,0.000000,0.199580,0.600041,0.779412,0.441464,19.808824,68,53
4,rf,10,0.250585,0.143957,0.177580,0.000000,0.157353,0.681414,0.838235,0.441464,19.808824,68,53


In [7]:
run_experiment('xgb', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: xgb (tfidf)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0    xgb   1  0.223140  0.080238      0.052164         0.044118   
1    xgb   3  0.275132  0.127441      0.076027         0.000000   
2    xgb   5  0.229572  0.116095      0.109878         0.000000   
3    xgb   7  0.193846  0.108027      0.145394         0.000000   
4    xgb  10  0.173302  0.116935      0.195893         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.397059     0.186141          0.397059  0.371136       25.220588   
1        0.254902     0.364439          0.558824  0.371136       25.220588   
2        0.173529     0.409677          0.632353  0.371136       25.220588   
3        0.132353     0.440314          0.661765  0.371136       25.220588   
4        0.108824     0.486889          0.705882  0.371136       25.220588   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3     

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,xgb,1,0.223140,0.080238,0.052164,0.044118,0.397059,0.186141,0.397059,0.371136,25.220588,68,53
1,xgb,3,0.275132,0.127441,0.076027,0.000000,0.254902,0.364439,0.558824,0.371136,25.220588,68,53
2,xgb,5,0.229572,0.116095,0.109878,0.000000,0.173529,0.409677,0.632353,0.371136,25.220588,68,53
3,xgb,7,0.193846,0.108027,0.145394,0.000000,0.132353,0.440314,0.661765,0.371136,25.220588,68,53
4,xgb,10,0.173302,0.116935,0.195893,0.000000,0.108824,0.486889,0.705882,0.371136,25.220588,68,53


In [8]:
run_experiment('mlp', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: mlp (tfidf)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0    mlp   1  0.190083  0.064384      0.054384         0.029412   
1    mlp   3  0.269841  0.101151      0.076582         0.000000   
2    mlp   5  0.268482  0.135080      0.104329         0.000000   
3    mlp   7  0.240000  0.124235      0.137070         0.000000   
4    mlp  10  0.222482  0.130428      0.184240         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.338235     0.145104          0.338235  0.366756       21.588235   
1        0.250000     0.307016          0.485294  0.366756       21.588235   
2        0.202941     0.429074          0.632353  0.366756       21.588235   
3        0.163866     0.476623          0.676471  0.366756       21.588235   
4        0.139706     0.601111          0.779412  0.366756       21.588235   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3     

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,mlp,1,0.190083,0.064384,0.054384,0.029412,0.338235,0.145104,0.338235,0.366756,21.588235,68,53
1,mlp,3,0.269841,0.101151,0.076582,0.000000,0.250000,0.307016,0.485294,0.366756,21.588235,68,53
2,mlp,5,0.268482,0.135080,0.104329,0.000000,0.202941,0.429074,0.632353,0.366756,21.588235,68,53
3,mlp,7,0.240000,0.124235,0.137070,0.000000,0.163866,0.476623,0.676471,0.366756,21.588235,68,53
4,mlp,10,0.222482,0.130428,0.184240,0.000000,0.139706,0.601111,0.779412,0.366756,21.588235,68,53


## Baseline - Word2Vec

In [9]:
# Obtain data prepared for WORD2VEC
embedding_type = 'word2vec'
X_train, X_test, y_train, y_test, classes = get_data_pipeline(embedding_type)
run_baselines(y_train, y_test, classes, embedding_type)

-> Generating Word2Vec features (dim=300)...
Pipeline completed. X_train shape: (270, 300), X_test shape: (68, 300)
                         method   k  micro_f1  macro_f1  hamming_loss  \
0       baseline_frequency_topk   1  0.181818  0.009224      0.054939   
1       baseline_frequency_topk   3  0.238095  0.019934      0.079911   
2       baseline_frequency_topk   5  0.229572  0.026978      0.109878   
3       baseline_frequency_topk   7  0.215385  0.032597      0.141509   
4       baseline_frequency_topk  10  0.187354  0.037825      0.192564   
5  baseline_random_distribution   1  0.157025  0.019749      0.056604   
6  baseline_random_distribution   3  0.164021  0.022607      0.087680   
7  baseline_random_distribution   5  0.175097  0.042265      0.117647   
8  baseline_random_distribution   7  0.175385  0.057509      0.148724   
9  baseline_random_distribution  10  0.156909  0.058278      0.199778   

   subset_accuracy  precision_at_k  recall_at_k  partial_hit_at_k      lrap  \
0

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,baseline_frequency_topk,1,0.181818,0.009224,0.054939,0.0,0.323529,0.129797,0.323529,0.344820,23.529412,68,53
1,baseline_frequency_topk,3,0.238095,0.019934,0.079911,0.0,0.220588,0.282878,0.397059,0.344820,23.529412,68,53
2,baseline_frequency_topk,5,0.229572,0.026978,0.109878,0.0,0.173529,0.392058,0.573529,0.344820,23.529412,68,53
3,baseline_frequency_topk,7,0.215385,0.032597,0.141509,0.0,0.147059,0.443284,0.632353,0.344820,23.529412,68,53
4,baseline_frequency_topk,10,0.187354,0.037825,0.192564,0.0,0.117647,0.499796,0.705882,0.344820,23.529412,68,53
5,baseline_random_distribution,1,0.157025,0.019749,0.056604,0.0,0.279412,0.114601,0.279412,0.262877,28.161765,68,53
6,baseline_random_distribution,3,0.164021,0.022607,0.087680,0.0,0.151961,0.196954,0.367647,0.262877,28.161765,68,53
7,baseline_random_distribution,5,0.175097,0.042265,0.117647,0.0,0.132353,0.259320,0.470588,0.262877,28.161765,68,53
8,baseline_random_distribution,7,0.175385,0.057509,0.148724,0.0,0.119748,0.333836,0.573529,0.262877,28.161765,68,53
9,baseline_random_distribution,10,0.156909,0.058278,0.199778,0.0,0.098529,0.395117,0.602941,0.262877,28.161765,68,53


## Machine Learning - Word2Vec

In [10]:
run_experiment('br_lr', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: br_lr (word2vec)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0  br_lr   1  0.074380  0.020445      0.062153              0.0   
1  br_lr   3  0.137566  0.041747      0.090455              0.0   
2  br_lr   5  0.132296  0.046878      0.123751              0.0   
3  br_lr   7  0.126154  0.057004      0.157603              0.0   
4  br_lr  10  0.133489  0.080722      0.205327              0.0   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.132353     0.057353          0.132353  0.216043       28.735294   
1        0.127451     0.177451          0.294118  0.216043       28.735294   
2        0.100000     0.217023          0.382353  0.216043       28.735294   
3        0.086134     0.248997          0.426471  0.216043       28.735294   
4        0.083824     0.352435          0.544118  0.216043       28.735294   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,br_lr,1,0.074380,0.020445,0.062153,0.0,0.132353,0.057353,0.132353,0.216043,28.735294,68,53
1,br_lr,3,0.137566,0.041747,0.090455,0.0,0.127451,0.177451,0.294118,0.216043,28.735294,68,53
2,br_lr,5,0.132296,0.046878,0.123751,0.0,0.100000,0.217023,0.382353,0.216043,28.735294,68,53
3,br_lr,7,0.126154,0.057004,0.157603,0.0,0.086134,0.248997,0.426471,0.216043,28.735294,68,53
4,br_lr,10,0.133489,0.080722,0.205327,0.0,0.083824,0.352435,0.544118,0.216043,28.735294,68,53


In [11]:
run_experiment('gb', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: gb (word2vec)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0     gb   1  0.181818  0.041443      0.054939         0.000000   
1     gb   3  0.243386  0.071614      0.079356         0.014706   
2     gb   5  0.237354  0.097158      0.108768         0.000000   
3     gb   7  0.230769  0.098738      0.138735         0.000000   
4     gb  10  0.206089  0.100374      0.188124         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.323529     0.127562          0.323529  0.331832       20.852941   
1        0.225490     0.287016          0.500000  0.331832       20.852941   
2        0.179412     0.384075          0.617647  0.331832       20.852941   
3        0.157563     0.479039          0.705882  0.331832       20.852941   
4        0.129412     0.575124          0.794118  0.331832       20.852941   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3   

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,gb,1,0.181818,0.041443,0.054939,0.000000,0.323529,0.127562,0.323529,0.331832,20.852941,68,53
1,gb,3,0.243386,0.071614,0.079356,0.014706,0.225490,0.287016,0.500000,0.331832,20.852941,68,53
2,gb,5,0.237354,0.097158,0.108768,0.000000,0.179412,0.384075,0.617647,0.331832,20.852941,68,53
3,gb,7,0.230769,0.098738,0.138735,0.000000,0.157563,0.479039,0.705882,0.331832,20.852941,68,53
4,gb,10,0.206089,0.100374,0.188124,0.000000,0.129412,0.575124,0.794118,0.331832,20.852941,68,53


In [12]:
run_experiment('rf', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: rf (word2vec)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0     rf   1  0.190083  0.023081      0.054384              0.0   
1     rf   3  0.248677  0.053312      0.078801              0.0   
2     rf   5  0.233463  0.061125      0.109323              0.0   
3     rf   7  0.236923  0.113638      0.137625              0.0   
4     rf  10  0.210773  0.114544      0.187014              0.0   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.338235     0.131974          0.338235  0.359092       22.132353   
1        0.230392     0.290938          0.455882  0.359092       22.132353   
2        0.176471     0.375392          0.529412  0.359092       22.132353   
3        0.161765     0.472183          0.647059  0.359092       22.132353   
4        0.132353     0.565565          0.764706  0.359092       22.132353   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3   

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,rf,1,0.190083,0.023081,0.054384,0.0,0.338235,0.131974,0.338235,0.359092,22.132353,68,53
1,rf,3,0.248677,0.053312,0.078801,0.0,0.230392,0.290938,0.455882,0.359092,22.132353,68,53
2,rf,5,0.233463,0.061125,0.109323,0.0,0.176471,0.375392,0.529412,0.359092,22.132353,68,53
3,rf,7,0.236923,0.113638,0.137625,0.0,0.161765,0.472183,0.647059,0.359092,22.132353,68,53
4,rf,10,0.210773,0.114544,0.187014,0.0,0.132353,0.565565,0.764706,0.359092,22.132353,68,53


In [13]:
run_experiment('xgb', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: xgb (word2vec)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0    xgb   1  0.198347  0.055909      0.053829         0.044118   
1    xgb   3  0.222222  0.095871      0.081576         0.000000   
2    xgb   5  0.214008  0.122887      0.112098         0.000000   
3    xgb   7  0.184615  0.114092      0.147059         0.000000   
4    xgb  10  0.175644  0.118836      0.195339         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.352941     0.161876          0.352941  0.329422       23.632353   
1        0.205882     0.273997          0.485294  0.329422       23.632353   
2        0.161765     0.333668          0.558824  0.329422       23.632353   
3        0.126050     0.370432          0.617647  0.329422       23.632353   
4        0.110294     0.459059          0.720588  0.329422       23.632353   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3  

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,xgb,1,0.198347,0.055909,0.053829,0.044118,0.352941,0.161876,0.352941,0.329422,23.632353,68,53
1,xgb,3,0.222222,0.095871,0.081576,0.000000,0.205882,0.273997,0.485294,0.329422,23.632353,68,53
2,xgb,5,0.214008,0.122887,0.112098,0.000000,0.161765,0.333668,0.558824,0.329422,23.632353,68,53
3,xgb,7,0.184615,0.114092,0.147059,0.000000,0.126050,0.370432,0.617647,0.329422,23.632353,68,53
4,xgb,10,0.175644,0.118836,0.195339,0.000000,0.110294,0.459059,0.720588,0.329422,23.632353,68,53


In [14]:
run_experiment('mlp', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: mlp (word2vec)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0    mlp   1  0.173554  0.009434      0.055494         0.000000   
1    mlp   3  0.238095  0.026975      0.079911         0.000000   
2    mlp   5  0.217899  0.034622      0.111543         0.014706   
3    mlp   7  0.206154  0.043045      0.143174         0.000000   
4    mlp  10  0.187354  0.059164      0.192564         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.308824     0.127696          0.308824  0.331628       23.308824   
1        0.220588     0.275392          0.441176  0.331628       23.308824   
2        0.164706     0.343039          0.529412  0.331628       23.308824   
3        0.140756     0.413248          0.602941  0.331628       23.308824   
4        0.117647     0.492421          0.691176  0.331628       23.308824   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3  

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,mlp,1,0.173554,0.009434,0.055494,0.000000,0.308824,0.127696,0.308824,0.331628,23.308824,68,53
1,mlp,3,0.238095,0.026975,0.079911,0.000000,0.220588,0.275392,0.441176,0.331628,23.308824,68,53
2,mlp,5,0.217899,0.034622,0.111543,0.014706,0.164706,0.343039,0.529412,0.331628,23.308824,68,53
3,mlp,7,0.206154,0.043045,0.143174,0.000000,0.140756,0.413248,0.602941,0.331628,23.308824,68,53
4,mlp,10,0.187354,0.059164,0.192564,0.000000,0.117647,0.492421,0.691176,0.331628,23.308824,68,53


## Baseline - Bert

In [15]:
# Obtain data prepared for BERT
embedding_type = 'bert'
X_train, X_test, y_train, y_test, classes = get_data_pipeline(embedding_type)
run_baselines(y_train, y_test, classes, embedding_type)

-> Generating BERT features (this may take some time)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33162.48it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


-> Generating BERT features (this may take some time)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 39793.40it/s]
[transformers] BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline completed. X_train shape: (270, 768), X_test shape: (68, 768)
                         method   k  micro_f1  macro_f1  hamming_loss  \
0       baseline_frequency_topk   1  0.181818  0.009224      0.054939   
1       baseline_frequency_topk   3  0.238095  0.019934      0.079911   
2       baseline_frequency_topk   5  0.229572  0.026978      0.109878   
3       baseline_frequency_topk   7  0.215385  0.032597      0.141509   
4       baseline_frequency_topk  10  0.187354  0.037825      0.192564   
5  baseline_random_distribution   1  0.157025  0.019749      0.056604   
6  baseline_random_distribution   3  0.164021  0.022607      0.087680   
7  baseline_random_distribution   5  0.175097  0.042265      0.117647   
8  baseline_random_distribution   7  0.175385  0.057509      0.148724   
9  baseline_random_distribution  10  0.156909  0.058278      0.199778   

   subset_accuracy  precision_at_k  recall_at_k  partial_hit_at_k      lrap  \
0              0.0        0.323529     0.12979

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,baseline_frequency_topk,1,0.181818,0.009224,0.054939,0.0,0.323529,0.129797,0.323529,0.344820,23.529412,68,53
1,baseline_frequency_topk,3,0.238095,0.019934,0.079911,0.0,0.220588,0.282878,0.397059,0.344820,23.529412,68,53
2,baseline_frequency_topk,5,0.229572,0.026978,0.109878,0.0,0.173529,0.392058,0.573529,0.344820,23.529412,68,53
3,baseline_frequency_topk,7,0.215385,0.032597,0.141509,0.0,0.147059,0.443284,0.632353,0.344820,23.529412,68,53
4,baseline_frequency_topk,10,0.187354,0.037825,0.192564,0.0,0.117647,0.499796,0.705882,0.344820,23.529412,68,53
5,baseline_random_distribution,1,0.157025,0.019749,0.056604,0.0,0.279412,0.114601,0.279412,0.262877,28.161765,68,53
6,baseline_random_distribution,3,0.164021,0.022607,0.087680,0.0,0.151961,0.196954,0.367647,0.262877,28.161765,68,53
7,baseline_random_distribution,5,0.175097,0.042265,0.117647,0.0,0.132353,0.259320,0.470588,0.262877,28.161765,68,53
8,baseline_random_distribution,7,0.175385,0.057509,0.148724,0.0,0.119748,0.333836,0.573529,0.262877,28.161765,68,53
9,baseline_random_distribution,10,0.156909,0.058278,0.199778,0.0,0.098529,0.395117,0.602941,0.262877,28.161765,68,53


## Bert

In [16]:
run_experiment('br_lr', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: br_lr (bert)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0  br_lr   1  0.239669  0.108288      0.051054         0.058824   
1  br_lr   3  0.396825  0.252189      0.063263         0.029412   
2  br_lr   5  0.361868  0.240749      0.091010         0.000000   
3  br_lr   7  0.326154  0.237104      0.121532         0.000000   
4  br_lr  10  0.278689  0.211575      0.170921         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.426471     0.198641          0.426471  0.496882       14.102941   
1        0.367647     0.500223          0.691176  0.496882       14.102941   
2        0.273529     0.607226          0.808824  0.496882       14.102941   
3        0.222689     0.674494          0.838235  0.496882       14.102941   
4        0.175000     0.737029          0.882353  0.496882       14.102941   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3    

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,br_lr,1,0.239669,0.108288,0.051054,0.058824,0.426471,0.198641,0.426471,0.496882,14.102941,68,53
1,br_lr,3,0.396825,0.252189,0.063263,0.029412,0.367647,0.500223,0.691176,0.496882,14.102941,68,53
2,br_lr,5,0.361868,0.240749,0.091010,0.000000,0.273529,0.607226,0.808824,0.496882,14.102941,68,53
3,br_lr,7,0.326154,0.237104,0.121532,0.000000,0.222689,0.674494,0.838235,0.496882,14.102941,68,53
4,br_lr,10,0.278689,0.211575,0.170921,0.000000,0.175000,0.737029,0.882353,0.496882,14.102941,68,53


In [17]:
run_experiment('gb', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: gb (bert)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0     gb   1  0.214876  0.092344      0.052719         0.073529   
1     gb   3  0.269841  0.111441      0.076582         0.000000   
2     gb   5  0.260700  0.123909      0.105438         0.000000   
3     gb   7  0.230769  0.123857      0.138735         0.000000   
4     gb  10  0.203747  0.123516      0.188679         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.382353     0.188592          0.382353  0.402823       21.220588   
1        0.250000     0.342408          0.544118  0.402823       21.220588   
2        0.197059     0.445980          0.676471  0.402823       21.220588   
3        0.157563     0.505784          0.735294  0.402823       21.220588   
4        0.127941     0.567934          0.764706  0.402823       21.220588   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3       

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,gb,1,0.214876,0.092344,0.052719,0.073529,0.382353,0.188592,0.382353,0.402823,21.220588,68,53
1,gb,3,0.269841,0.111441,0.076582,0.000000,0.250000,0.342408,0.544118,0.402823,21.220588,68,53
2,gb,5,0.260700,0.123909,0.105438,0.000000,0.197059,0.445980,0.676471,0.402823,21.220588,68,53
3,gb,7,0.230769,0.123857,0.138735,0.000000,0.157563,0.505784,0.735294,0.402823,21.220588,68,53
4,gb,10,0.203747,0.123516,0.188679,0.000000,0.127941,0.567934,0.764706,0.402823,21.220588,68,53


In [18]:
run_experiment('rf', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: rf (bert)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0     rf   1  0.280992  0.065874      0.048280         0.058824   
1     rf   3  0.407407  0.185753      0.062153         0.000000   
2     rf   5  0.381323  0.223782      0.088235         0.000000   
3     rf   7  0.320000  0.189514      0.122642         0.000000   
4     rf  10  0.271663  0.178922      0.172586         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.500000     0.234075          0.500000  0.523532       16.602941   
1        0.377451     0.517443          0.779412  0.523532       16.602941   
2        0.288235     0.618319          0.823529  0.523532       16.602941   
3        0.218487     0.641224          0.852941  0.523532       16.602941   
4        0.170588     0.715845          0.882353  0.523532       16.602941   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3       

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,rf,1,0.280992,0.065874,0.048280,0.058824,0.500000,0.234075,0.500000,0.523532,16.602941,68,53
1,rf,3,0.407407,0.185753,0.062153,0.000000,0.377451,0.517443,0.779412,0.523532,16.602941,68,53
2,rf,5,0.381323,0.223782,0.088235,0.000000,0.288235,0.618319,0.823529,0.523532,16.602941,68,53
3,rf,7,0.320000,0.189514,0.122642,0.000000,0.218487,0.641224,0.852941,0.523532,16.602941,68,53
4,rf,10,0.271663,0.178922,0.172586,0.000000,0.170588,0.715845,0.882353,0.523532,16.602941,68,53


In [19]:
run_experiment('xgb', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: xgb (bert)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0    xgb   1  0.272727  0.118772      0.048835         0.058824   
1    xgb   3  0.354497  0.181789      0.067703         0.000000   
2    xgb   5  0.330739  0.207929      0.095450         0.000000   
3    xgb   7  0.298462  0.209082      0.126526         0.000000   
4    xgb  10  0.264637  0.195671      0.174251         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.485294     0.218879          0.485294  0.464943       17.867647   
1        0.328431     0.409193          0.661765  0.464943       17.867647   
2        0.250000     0.498303          0.750000  0.464943       17.867647   
3        0.203782     0.576875          0.808824  0.464943       17.867647   
4        0.166176     0.677365          0.867647  0.464943       17.867647   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3      

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,xgb,1,0.272727,0.118772,0.048835,0.058824,0.485294,0.218879,0.485294,0.464943,17.867647,68,53
1,xgb,3,0.354497,0.181789,0.067703,0.000000,0.328431,0.409193,0.661765,0.464943,17.867647,68,53
2,xgb,5,0.330739,0.207929,0.095450,0.000000,0.250000,0.498303,0.750000,0.464943,17.867647,68,53
3,xgb,7,0.298462,0.209082,0.126526,0.000000,0.203782,0.576875,0.808824,0.464943,17.867647,68,53
4,xgb,10,0.264637,0.195671,0.174251,0.000000,0.166176,0.677365,0.867647,0.464943,17.867647,68,53


In [20]:
run_experiment('mlp', X_train, y_train, X_test, y_test, classes, embedding_type)


Running: mlp (bert)
  method   k  micro_f1  macro_f1  hamming_loss  subset_accuracy  \
0    mlp   1  0.280992  0.120250      0.048280         0.088235   
1    mlp   3  0.349206  0.159840      0.068257         0.000000   
2    mlp   5  0.299611  0.159263      0.099889         0.000000   
3    mlp   7  0.270769  0.170160      0.131521         0.000000   
4    mlp  10  0.250585  0.185219      0.177580         0.000000   

   precision_at_k  recall_at_k  partial_hit_at_k      lrap  coverage_error  \
0        0.500000     0.243389          0.500000  0.482561           17.25   
1        0.323529     0.429453          0.676471  0.482561           17.25   
2        0.226471     0.523081          0.764706  0.482561           17.25   
3        0.184874     0.580434          0.794118  0.482561           17.25   
4        0.157353     0.683486          0.823529  0.482561           17.25   

   n_samples  n_labels  
0         68        53  
1         68        53  
2         68        53  
3      

,method,k,micro_f1,macro_f1,hamming_loss,subset_accuracy,precision_at_k,recall_at_k,partial_hit_at_k,lrap,coverage_error,n_samples,n_labels
0,mlp,1,0.280992,0.120250,0.048280,0.088235,0.500000,0.243389,0.500000,0.482561,17.25,68,53
1,mlp,3,0.349206,0.159840,0.068257,0.000000,0.323529,0.429453,0.676471,0.482561,17.25,68,53
2,mlp,5,0.299611,0.159263,0.099889,0.000000,0.226471,0.523081,0.764706,0.482561,17.25,68,53
3,mlp,7,0.270769,0.170160,0.131521,0.000000,0.184874,0.580434,0.794118,0.482561,17.25,68,53
4,mlp,10,0.250585,0.185219,0.177580,0.000000,0.157353,0.683486,0.823529,0.482561,17.25,68,53


### Partial Hit Results by Top-K

Below we present the percentage of courses that had at least one correctly predicted competency within the *K* highest probabilities indicated by the model.

#### Table 1: Precision @ Top-1 (Accuracy of the 1st choice)

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | 39.71% | 30.88% | 48.53% |
| **XGBoost** | 32.35% | 36.76% | 45.59% |
| **MLP (Deep Learning)** | 29.41% | 29.41% | **50.00%** |
| **Gradient Boosting** | 29.41% | 30.88% | 33.82% |

#### Table 2: Accuracy @ Top-3

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | 61.76% | 47.06% | **66.18%** |
| **XGBoost** | 58.82% | 54.41% | **66.18%** |
| **MLP (Deep Learning)** | 54.41% | 38.24% | 61.76% |

| **Gradient Boosting** | 55.88% | 52.94% | 55.88% |

#### Table 3: Accuracy @ Top-5

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | 73.53% | 60.29% | **80.88%** |
| **XGBoost** | 60.29% | 64.71% | 73.53% |
| **MLP (Deep Learning)** | 64.71% | 52.94% | 73.53% |
| **Gradient Boosting** | 64.71% | 61.76% | 67.65% |

#### Table 4: Accuracy @ Top-7

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | 79.41% | 72.06% | **83.82%** |
| **XGBoost** | 69.12% | 66.18% | 80.88% |
| **MLP (Deep Learning)** | 69.12% | 60.29% | 76.47% |
| **Gradient Boosting** | 75.00% | 66.18% | 73.53% |

#### Table 5: Accuracy @ Top-10

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **86.76%** | 72.06% | **86.76%** |
| **XGBoost** | 72.06% | 75.00% | **86.76%** |
| **MLP (Deep Learning)** | 76.47% | 67.65% | 82.35% |
| **Gradient Boosting** | 80.88% | 76.47% | 79.41% |

## Precision Results @ Top-K

The Precision@K metric indicates the average proportion of correctly predicted competencies among the *K* competencies suggested by the model for each course.

### Table 1 — Precision @ Top-1

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.3971** | 0.3088 | **0.4853** |
| **XGBoost** | 0.3235 | 0.3676 | 0.4559 |
| **MLP (Deep Learning)** | 0.2941 | 0.2941 | 0.4559 |
| **Gradient Boosting** | 0.2941 | 0.3088 | 0.3382 |

### Table 2 — Accuracy @ Top-3

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.2892** | 0.2304 | **0.3382** |
| **XGBoost** | 0.2745 | **0.2745** | 0.3284 |
| **MLP (Deep Learning)** | 0.2549 | 0.1863 | **0.3480** |
| **Gradient Boosting** | 0.2500 | 0.2353 | 0.2598 |

### Table 3 — Accuracy @ Top-5

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.2382** | 0.1765 | **0.2794** |
| **XGBoost** | 0.1912 | **0.2029** | 0.2471 |
| **MLP (Deep Learning)** | 0.1971 | 0.1559 | 0.2618 |
| **Gradient Boosting** | 0.1853 | 0.1765 | 0.1971 |

### Table 4 — Accuracy @ Top-7

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.1954** | **0.1576** | **0.2206** |
| **XGBoost** | 0.1534 | 0.1576 | 0.2059 |
| **MLP (Deep Learning)** | 0.1597 | 0.1387 | 0.2059 |
| **Gradient Boosting** | 0.1660 | 0.1366 | 0.1618 |

### Table 5 — Accuracy @ Top-10

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.1603** | 0.1368 | **0.1750** | | **XGBoost** | 0.1221 | 0.1235 | 0.1662 |
| **MLP (Deep Learning)** | 0.1353 | 0.1206 | 0.1603 |
| **Gradient Boosting** | 0.1338 | 0.1221 | 0.1338 |

> Unlike the hit rate metric, Precision@K explicitly penalizes the inclusion of incorrect skills, offering a more rigorous assessment of the quality of the lists generated by the models.